# Cortina Generator — Test Notebook
Tests each layer: tanda summarizer → prompt crafter → Lyria audio → full generate_cortina.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv('../.env')

from atdj.config import GEMINI_API_KEY
print('GEMINI_API_KEY set:', bool(GEMINI_API_KEY))

GEMINI_API_KEY set: True


## 1 — Test `_summarize_tanda`

In [2]:
from atdj.cortina.generator import _summarize_tanda

mock_tracks = [
    {'style': 'tango', 'orchestra': 'Di Sarli', 'decade': '1940s', 'energy': 0.6, 'bpm': 124},
    {'style': 'tango', 'orchestra': 'Di Sarli', 'decade': '1940s', 'energy': 0.7, 'bpm': 128},
    {'style': 'tango', 'orchestra': 'Di Sarli', 'decade': '1940s', 'energy': 0.65, 'bpm': 126},
]

summary = _summarize_tanda(mock_tracks)
print(summary)

{'style': 'tango', 'orchestra': 'Di Sarli', 'decade': '1940s', 'energy_label': 'moderate', 'avg_bpm': 126, 'mood': 'expressive and danceable'}


## 2 — Test `_craft_music_prompt` (uses Gemini LLM)

In [3]:
from atdj.cortina.generator import _craft_music_prompt

prev = _summarize_tanda(mock_tracks)

# Closing cortina (next_style=None)
closing_prompt = _craft_music_prompt(prev, next_style=None)
print('=== Closing cortina prompt ===')
print(closing_prompt)

2026-05-03 21:17:27.097 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 21:17:27.097 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`
2026-05-03 21:17:27.097 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 21:17:27.097 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 21:17:27.097 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 21:17:27.098 WARNING streamlit.runtime.scriptrunner_utils.script_run_c

=== Closing cortina prompt ===
Generate a 25-second instrumental Bossa Nova cortina with a smooth, mid-tempo groove at approximately 126 BPM. Feature a gentle acoustic guitar rhythm, a walking upright bass line, light brushed drums, and a melodic Rhodes piano, creating an effortlessly sophisticated and flowing atmosphere.


## 3 — Test `_call_lyria` (generates audio)

In [4]:
from pathlib import Path
from atdj.cortina.generator import _call_lyria

out_dir = Path('../data/cortinas/generated')
out_dir.mkdir(parents=True, exist_ok=True)

audio_path = _call_lyria(closing_prompt, out_dir / 'test_cortina', GEMINI_API_KEY)
print('Audio saved to:', audio_path)
print('File size:', audio_path.stat().st_size, 'bytes')

Audio saved to: ../data/cortinas/generated/test_cortina.mp3
File size: 744607 bytes


## 4 — Play the audio

In [5]:
from IPython.display import Audio
Audio(str(audio_path))

## 5 — Test full `generate_cortina`

In [6]:
from atdj.cortina.generator import generate_cortina

result = generate_cortina(
    prev_tracks=mock_tracks,
    next_style=None,
    output_dir=out_dir,
    api_key=GEMINI_API_KEY,
)
print(result)
Audio(result['file_path'])

2026-05-03 21:18:20.647 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 21:18:20.647 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 21:18:20.648 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 21:18:20.648 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 21:18:20.649 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


{'type': 'cortina', 'title': 'Cortina (tango)', 'file_path': '../data/cortinas/generated/cortina_02d3acde.mp3', 'duration': '0:25', 'source': 'generated', 'music_prompt': 'Generate a 25-second cool jazz cortina with a moderate energy level, around 126 BPM. Feature a smooth melody from a tenor saxophone over a walking bass line, light swinging drums, and sophisticated piano chords, creating a laid-back yet engaging atmosphere without any tango elements.'}
